# Week 2, Notebook B: train and evaluate the baseline



In [ ]:
# Importing libraries, explicit paths and settings
import hashlib
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

# Explicit project folder. Edit this one line if your Week_2 folder is somewhere else.
PROJECT_DIR = Path.home() / "Downloads" / "main" / "Week_2"

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "week2_dataset.csv"
VECTORS_PATH = OUTPUT_DIR / "vectors_xlm-roberta-base.npz"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"
PER_CLASS_PATH = OUTPUT_DIR / "per_class_metrics.csv"
RUN_SETTINGS_PATH = OUTPUT_DIR / "run_settings.json"

# settings 
MODEL_NAME = "xlm-roberta-base"
TRAIN_LANGUAGES = ["hi"]          # the classifiers learn from these languages 
MAX_TOKENS = 128                  # longer reviews are cut after this many tokens
EMBED_BATCH_SIZE = 16             # reviews processed at once while computing vectors (lower it if memory is short)
C_VALUES = [0.01, 0.1, 1, 10]     # classifier settings to try; the best one on the validation reviews is used
SEED = 42

In [ ]:
# Loading and validating the prepared dataset
dataset = pd.read_csv(DATASET_PATH)

required = ["review_id", "language", "text", "part", "label", "label_id", "intent", "intent_id", "unsure"]
missing = [c for c in required if c not in dataset.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Run Notebook A again.")
if dataset[required].isna().any().any():
    raise ValueError("Dataset contains missing values.")
if set(dataset["language"]) != {"hi", "bn", "mr"}:
    raise ValueError("Expected Hindi, Bengali, and Marathi.")
if set(dataset["part"]) != {"train", "val", "test"}:
    raise ValueError("Expected train, val, and test partitions.")
if not set(TRAIN_LANGUAGES) <= set(dataset["language"]):
    raise ValueError(f"TRAIN_LANGUAGES {TRAIN_LANGUAGES} are not all in the dataset.")
if dataset.duplicated(["review_id", "language"]).any():
    raise ValueError("Duplicate review-language rows found.")
if dataset.groupby("review_id")["part"].nunique().gt(1).any():
    raise ValueError("A review appears in multiple partitions.")

# each partition must contain the same reviews in all languages
reviews_in = dataset.groupby(["part", "language"])["review_id"].apply(set)
for part in ["train", "val", "test"]:
    if not (reviews_in[(part, "hi")] == reviews_in[(part, "bn")] == reviews_in[(part, "mr")]):
        raise ValueError(f"Review IDs are not aligned across languages in {part}.")

# the classes that really exist in the data
tasks = {
    "sentiment": ("label_id", dict(zip(dataset["label_id"], dataset["label"]))),
    "intent": ("intent_id", dict(zip(dataset["intent_id"], dataset["intent"]))),
}

print(f"Rows: {len(dataset)} | Unique reviews: {dataset['review_id'].nunique()}")
display(dataset.groupby(["language", "part"]).size().unstack())

hindi = dataset[dataset["language"] == "hi"]
for task, (column, names) in tasks.items():
    counts = pd.crosstab(hindi["part"], hindi[column].map(names))
    print(f"\n{task}: {len(names)} classes with examples -> {list(names.values())}")
    display(counts)
    if (counts == 0).any().any():
        print(f"WARNING: a {task} class has NO reviews in some part, so it cannot be learned or evaluated there.")
    elif (counts < 10).any().any():
        print(f"WARNING: some {task} classes have fewer than 10 reviews in a part. Scores for those classes are not reliable.")
print("\nClasses with no examples at all (Neutral sentiment, Inquiry intent) are not in the data, so they cannot be learned or tested.")
print("Dataset validation passed.")

Rows: 3456 | Unique reviews: 1152


part,test,train,val
language,,,
bn,173,806,173
hi,173,806,173
mr,173,806,173



sentiment: 2 classes with examples -> ['Negative', 'Positive']


label_id,Negative,Positive
part,,
test,86,87
train,400,406
val,86,87



intent: 2 classes with examples -> ['Factual', 'Opinion']


intent_id,Factual,Opinion
part,,
test,13,160
train,66,740
val,16,157



Classes with no examples at all (Neutral sentiment, Inquiry intent) are not in the data, so they cannot be learned or tested.
Dataset validation passed.


In [ ]:
#Encoder, device and vectors
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()      # Apple Silicon Macs
    else "cpu"
)
print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | Device: {device}")

texts = dataset["text"].tolist()
input_sha256 = hashlib.sha256(
    json.dumps(dataset[["review_id", "language", "text"]].values.tolist(), ensure_ascii=False).encode("utf-8")
).hexdigest()

vectors = None
if VECTORS_PATH.is_file():
    with np.load(VECTORS_PATH) as saved:
        if (
            {"vectors", "model_name", "max_tokens", "input_sha256"} <= set(saved.files)
            and str(saved["model_name"]) == MODEL_NAME
            and int(saved["max_tokens"]) == MAX_TOKENS
            and str(saved["input_sha256"]) == input_sha256
            and saved["vectors"].ndim == 2
            and len(saved["vectors"]) == len(dataset)
        ):
            vectors = saved["vectors"].copy()
            print(f"Using cached vectors: {vectors.shape}")
        else:
            print("Cache does not match; recomputing embeddings.")

if vectors is None:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()      # used as it is: the encoder is not trained

    lengths = [len(ids) for ids in tokenizer(texts)["input_ids"]]
    print(f"Texts truncated at {MAX_TOKENS} tokens: {(np.array(lengths) > MAX_TOKENS).mean() * 100:.1f}%")

    batches = []
    with torch.no_grad():                                                  # no training, so no gradients are needed
        for start in range(0, len(texts), EMBED_BATCH_SIZE):
            batch = tokenizer(
                texts[start:start + EMBED_BATCH_SIZE],
                truncation=True, max_length=MAX_TOKENS, padding=True, return_tensors="pt",
            ).to(device)
            token_vectors = encoder(**batch).last_hidden_state             # shape: (texts, tokens, 768)
            mask = batch["attention_mask"].unsqueeze(-1)                   # 1 for real tokens, 0 for padding
            batches.append(((token_vectors * mask).sum(dim=1) / mask.sum(dim=1)).cpu().numpy())

    vectors = np.vstack(batches)
    np.savez(VECTORS_PATH, vectors=vectors, model_name=MODEL_NAME, max_tokens=MAX_TOKENS, input_sha256=input_sha256)
    print(f"Saved cached vectors: {VECTORS_PATH.name} | Shape: {vectors.shape}")

PyTorch: 2.8.0 | Transformers: 4.57.6 | Device: mps
Using cached vectors: (3456, 768)


In [59]:
# Train the classifiers (train part) and choose `C` (validation part)

def score_rows(true_ids, predicted_ids, class_ids):
    """Accuracy, macro-F1 and the numbers per class. Macro-F1 is the average F1 over the classes that have examples."""
    true_ids, predicted_ids = np.asarray(true_ids), np.asarray(predicted_ids)
    precision, recall, f1, support = precision_recall_fscore_support(true_ids, predicted_ids, labels=class_ids, zero_division=0)
    return float((true_ids == predicted_ids).mean()), float(f1[support > 0].mean()), precision, recall, f1, support

train_rows = (dataset["part"] == "train") & dataset["language"].isin(TRAIN_LANGUAGES)
select_rows = (dataset["part"] == "val") & dataset["language"].isin(TRAIN_LANGUAGES)
print("training reviews:", int(train_rows.sum()), "| validation reviews used to choose C:", int(select_rows.sum()))

# (model, task) -> predicted class numbers for EVERY row of the dataset.
# "majority-class" is a model that ignores the text and always answers the most common training class.
predictions = {("majority-class", task): np.full(len(dataset), dataset.loc[train_rows, column].mode()[0])
               for task, (column, names) in tasks.items()}
chosen_settings = {}

for task, (column, names) in tasks.items():
    if dataset.loc[train_rows, column].nunique() < 2:
        raise ValueError(f"the training data has only one {task} class, so nothing can be learned")

    best = None
    for C in C_VALUES:
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=SEED))
        classifier.fit(vectors[train_rows.values], dataset.loc[train_rows, column])
        _, macro_f1, *_ = score_rows(dataset.loc[select_rows, column], classifier.predict(vectors[select_rows.values]), sorted(names))
        print(f"{MODEL_NAME} | {task} | C={C}: macro-F1 on validation = {macro_f1:.3f}")
        if best is None or macro_f1 > best[0]:                # on a tie the smaller (simpler) C is kept
            best = (macro_f1, C, classifier)
    macro_f1, C, classifier = best
    print(f"--> {MODEL_NAME} | {task}: chosen C = {C}")
    predictions[(MODEL_NAME, task)] = classifier.predict(vectors)
    chosen_settings[f"{MODEL_NAME} | {task}"] = {"C": C, "validation_macro_f1": round(macro_f1, 4)}

training reviews: 806 | validation reviews used to choose C: 173
xlm-roberta-base | sentiment | C=0.01: macro-F1 on validation = 0.925
xlm-roberta-base | sentiment | C=0.1: macro-F1 on validation = 0.902
xlm-roberta-base | sentiment | C=1: macro-F1 on validation = 0.896
xlm-roberta-base | sentiment | C=10: macro-F1 on validation = 0.884
--> xlm-roberta-base | sentiment: chosen C = 0.01
xlm-roberta-base | intent | C=0.01: macro-F1 on validation = 0.660
xlm-roberta-base | intent | C=0.1: macro-F1 on validation = 0.687
xlm-roberta-base | intent | C=1: macro-F1 on validation = 0.677
xlm-roberta-base | intent | C=10: macro-F1 on validation = 0.626
--> xlm-roberta-base | intent: chosen C = 0.1


/Users/madhavshahi/Downloads/main/.venv39/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/madhavshahi/Downloads/main/.venv39/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/madhavshahi/Downloads/main/.venv39/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/madhavshahi/Downloads/main/.venv39/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/madhavshahi/Downloads/main/.venv39/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountere

In [ ]:
# Score the validation and test reviews
subsets = {"all reviews": np.ones(len(dataset), dtype=bool),
           "only reviews not flagged unsure": (dataset["unsure"] == 0).values}

metric_rows, class_rows = [], []
for (model_name, task), predicted in predictions.items():
    column, names = tasks[task]
    class_ids = sorted(names)
    for part in ["val", "test"]:
        for language in ["hi", "bn", "mr"]:
            for subset_name, subset_mask in subsets.items():
                rows = ((dataset["part"] == part) & (dataset["language"] == language)).values & subset_mask
                predicted_ids = predicted[rows]
                accuracy, macro_f1, precision, recall, f1, support = score_rows(dataset.loc[rows, column].values, predicted_ids, class_ids)
                where = dict(model=model_name, task=task, part=part, language=language, subset=subset_name)
                metric_rows.append({**where, "n": int(rows.sum()), "accuracy": accuracy, "macro_f1": macro_f1,
                                    "classes_with_examples": int((support > 0).sum())})
                for i, class_id in enumerate(class_ids):
                    class_rows.append({**where, "class": names[class_id], "precision": precision[i], "recall": recall[i],
                                       "f1": f1[i], "support": int(support[i]), "predicted_as_this_class": int((predicted_ids == class_id).sum())})

metrics = pd.DataFrame(metric_rows)
per_class = pd.DataFrame(class_rows)
print("scored", len(metrics), "combinations")
no_examples = per_class[per_class["support"] == 0]
if len(no_examples) > 0:
    print("WARNING: these classes have no examples in the evaluated reviews, so they were not evaluated there:")
    display(no_examples[["model", "task", "part", "language", "subset", "class"]].drop_duplicates())

scored 48 combinations


In [ ]:
#Saving the metrics and the run settings
metrics.to_csv(METRICS_PATH, index=False)
per_class.to_csv(PER_CLASS_PATH, index=False)

settings = {
    "model": MODEL_NAME,
    "train_languages": TRAIN_LANGUAGES,
    "max_tokens": MAX_TOKENS,
    "c_values_tried": C_VALUES,
    "chosen": chosen_settings,
    "seed": SEED,
    "selection_rule": "C selected by validation macro-F1 using training-language reviews only; test reviews are not used for model selection.",
    "dataset_file": DATASET_PATH.name,
    "dataset_rows": len(dataset),
    "dataset_sha256": hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest(),
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    },
    "device": device,
}
RUN_SETTINGS_PATH.write_text(json.dumps(settings, indent=2), encoding="utf-8")

print("Saved in outputs/:", METRICS_PATH.name, PER_CLASS_PATH.name, RUN_SETTINGS_PATH.name)

Saved in outputs/: metrics.csv per_class_metrics.csv run_settings.json


In [ ]:
#Results
test_scores = metrics[(metrics["part"] == "test") & (metrics["subset"] == "all reviews")]
for task in tasks:
    table = test_scores[test_scores["task"] == task].pivot_table(index="model", columns="language", values=["macro_f1", "accuracy"])
    table = table.reindex(columns=[(m, l) for m in ["macro_f1", "accuracy"] for l in ["hi", "bn", "mr"]])
    print(f"\n{task.upper()} (test reviews; hi = trained language, bn and mr = never seen in training)")
    display(table.round(3))
print("\nNumber of test reviews per language:", int(test_scores["n"].iloc[0]))
print("Read macro-F1 first (intent is very imbalanced), and compare every model with the 'majority-class' row.")
print("Per-class results: per_class_metrics.csv")


SENTIMENT (test reviews; hi = trained language, bn and mr = never seen in training)


macro_f1               accuracy              
language               hi     bn     mr       hi     bn     mr
model                                                         
majority-class      0.335  0.335  0.335    0.503  0.503  0.503
xlm-roberta-base    0.942  0.878  0.884    0.942  0.879  0.884


INTENT (test reviews; hi = trained language, bn and mr = never seen in training)


macro_f1               accuracy              
language               hi     bn     mr       hi     bn     mr
model                                                         
majority-class      0.480  0.480  0.480    0.925  0.925  0.925
xlm-roberta-base    0.697  0.627  0.536    0.925  0.803  0.913


Number of test reviews per language: 173
Read macro-F1 first (intent is very imbalanced), and compare every model with the 'majority-class' row.
Per-class results: per_class_metrics.csv
